# PaddleOCR - forward half, paired with PaddleOCR_Kaggle_Reverse.ipynb

Run the two updated notebooks in **two independent Kaggle CPU sessions**, each with four available CPUs. Select Accelerator None, enable Internet, then Run All. This notebook defaults to RUN_PARTITION="forward", MODE="full", AUTO_RESTORE_VALIDATED=False for a fresh run. The reverse notebook defaults to RUN_PARTITION="reverse". Do not pair the reverse notebook with an older forward notebook that still selects all 9,000 pages.

Each region is split at its midpoint: forward selects the first 3,000 text and first 1,500 nontext pages in the canonical sorted ID lists; reverse selects the last 3,000 text and last 1,500 nontext pages and processes its list in descending order. Each session owns **4,500 pages**, with no overlap and combined coverage of 6,000 text + 3,000 nontext. This changes the order of documents, not the reading order or orientation inside a document. Four workers retain the original global page-index modulo assignment, so checkpoint worker IDs remain stable.

Forward output: `/kaggle/working/oprb_results/forward`, archive `/kaggle/working/oprb_results/forward_checkpoints.zip`. Reverse output uses `reverse` and `reverse_checkpoints.zip`. Both runs retain model revisions, preprocessing, OCR options, CPU settings, per-page timing, normalization and macro scoring. The 4x1/off configuration and exact package lock come from the successful Kaggle pilot. Known-failing MKL-DNN tuning is not repeated. Session supervision, bounded retries, 11-hour budget and SQLite snapshots remain enabled. A half can still take more than one session; parallelism does not remove Kaggle's time limit.

Download/save the ZIP from **each session separately**. Extract them into separate local forward/reverse directories; their worker SQLite basenames are intentionally identical and must not overwrite one another. Run `evnv/Scripts/python.exe merge_paddle_checkpoints.py --forward PATH_TO_FORWARD --reverse PATH_TO_REVERSE --output results/paddle_parallel_merged`. The merger checks matching configs, split manifests, row uniqueness, global worker assignment and coverage, then creates four merged worker databases. It never changes predictions, timings or scores; partial coverage is explicitly reported. Run the common evaluator on the four merged SQLite files for final scoring. Keep merge_provenance.json to report execution times separately by source session/partition rather than attributing them to one physical host.

To resume a half, attach that half's latest cumulative output and set AUTO_RESTORE_VALIDATED=True. Keep its directory name (`forward` or `reverse`) and all four SQLite filenames. Cross-partition restoration is rejected. For an intentional single-session full run only, set RUN_PARTITION="all"; do not pair that setting with another concurrent run.


In [ ]:
from pathlib import Path
import sys, os, subprocess, tempfile, json
MODE = "full"  # The uploaded four-worker pilot succeeded; continue all 9,000 pages
FULL_SLICE = slice(None)
if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Open this notebook on Kaggle with Internet enabled (CPU runtime).")
RUN_PARTITION = "forward"  # forward, reverse, or all; use forward + reverse in two separate Kaggle sessions
OUTPUT = Path("/kaggle/working/oprb_results") / RUN_PARTITION
GPU_WORKERS = 1
RESUME_FROM = None  # New CPU experiment: do not restore the previous 12-hour checkpoint.
CPU_TUNE_PAGES_PER_REGION = 1  # calibration: 2 text + 1 nontext page, repeated below
CPU_TUNE_REPEATS = 2
CPU_PROGRESS_EVERY = 25
CPU_TUNE_TIMEOUT_SECONDS = 900
# cpu_tuning.json is reused only when hardware, packages, weights and reader match.
# Preserve the budget across Run All/kernel restarts in the same Kaggle host session.
session_clock = Path("/kaggle/working/.oprb_session_clock.json")
boot_path = Path("/proc/sys/kernel/random/boot_id")
boot_id = boot_path.read_text().strip() if boot_path.is_file() else None
now_monotonic = __import__("time").monotonic()
try:
    saved_clock = json.loads(session_clock.read_text())
except (OSError, ValueError):
    saved_clock = {}
SESSION_STARTED = saved_clock.get("started", now_monotonic)
if saved_clock.get("boot_id") != boot_id or not isinstance(SESSION_STARTED, (int,float)) or SESSION_STARTED > now_monotonic:
    SESSION_STARTED = now_monotonic
session_clock.write_text(json.dumps(dict(started=SESSION_STARTED, boot_id=boot_id)))
SESSION_BUDGET_HOURS = 11.0  # Includes setup; leave time to save outputs before Kaggle's 12-hour limit.
MAX_WORKER_RESTARTS = 2
WORKER_STALL_SECONDS = 900
AUTO_RESTORE_VALIDATED = False  # New parallel run; enable only with this partition's latest output attached
VALIDATED_CONFIG = {'cpu': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'cpu_parallelism': 'independent page processes; shared host resources', 'dataset': 'c7ab9fdf2bf33f297bba62e4842d3e5094d11ec2', 'device': 'CPU', 'enable_mkldnn': False, 'model': 'paddle', 'opencv_threads': 1, 'precision': 'float32', 'prompt': '', 'reader_sha256': 'e706489c9e65149c594545522fdf3f9860cdb2c3e3197b8e39d4117235c24a7d', 'revisions': {'PaddlePaddle/PP-OCRv5_mobile_det': '0d63e78e2b680928f6b1747d76a08db6e645efb7', 'PaddlePaddle/PP-OCRv5_mobile_rec': '682f20538d8c086cb2128e5cfac775e6c4904e85'}, 'thread_limits': 'OMP/MKL/OPENBLAS/NUMEXPR=threads_per_worker', 'threads': 1, 'timing_mode': 'single_page', 'versions': {'aiohappyeyeballs': '2.7.1', 'aiohttp': '3.14.5', 'aiosignal': '1.4.0', 'aistudio_sdk': '0.3.9', 'annotated-types': '0.8.0', 'anyio': '4.16.0', 'asttokens': '3.0.2', 'attrs': '26.1.0', 'autocommand': '2.2.2', 'backports.tarfile': '1.2.0', 'bce-python-sdk': '0.9.79', 'certifi': '2026.11.20', 'cffi': '2.1.1', 'chardet': '7.7.0', 'charset-normalizer': '3.5.2', 'click': '8.5.1', 'colorlog': '6.12.0', 'comm': '0.2.3', 'crc32c': '2.9.post0', 'cryptography': '50.0.2', 'debugpy': '1.8.22', 'executing': '2.3.0', 'filelock': '4.3.0', 'frozenlist': '1.8.0', 'fsspec': '2026.9.0', 'future': '1.0.0', 'h11': '0.16.0', 'hf-xet': '1.8.0', 'httpcore': '1.0.9', 'httpx': '0.28.1', 'huggingface_hub': '1.33.0', 'idna': '3.20', 'imagesize': '2.0.1', 'importlib_metadata': '8.7.1', 'ipykernel': '7.4.0', 'ipython': '9.18.1', 'ipython_pygments_lexers': '1.1.1', 'jaraco.context': '6.1.0', 'jaraco.functools': '4.4.0', 'jaraco.text': '4.0.0', 'jedi': '0.20.1', 'jiwer': '4.0.0', 'jupyter_client': '8.10.0', 'jupyter_core': '5.9.1', 'matplotlib-inline': '0.2.2', 'modelscope': '1.40.3', 'modelscope-hub': '0.7.1', 'more-itertools': '10.8.0', 'multidict': '7.2.1', 'nest-asyncio2': '1.7.4', 'networkx': '3.7', 'numpy': '2.3.5', 'opencv-contrib-python': '4.10.0.84', 'opt-einsum': '3.3.0', 'packaging': '26.3', 'paddleocr': '3.7.0', 'paddlepaddle': '3.3.1', 'paddlex': '3.7.2', 'pandas': '3.1.1', 'parso': '0.8.7', 'pexpect': '4.9.0', 'pillow': '13.0.0', 'platformdirs': '4.4.0', 'prettytable': '3.19.0', 'prompt_toolkit': '3.0.53', 'propcache': '0.5.4', 'protobuf': '7.36.2', 'psutil': '7.2.2', 'ptyprocess': '0.7.0', 'pure_eval': '0.2.4', 'py-cpuinfo': '9.0.0', 'pyclipper': '1.4.0', 'pycparser': '3.12', 'pycryptodome': '4.5.0', 'pydantic': '2.14.4', 'pydantic_core': '2.50.5', 'pygments': '2.21.0', 'pypdfium2': '5.15.2', 'python-bidi': '0.6.12', 'python-dateutil': '4.0.0', 'pyyaml': '6.0.2', 'pyzmq': '27.3.0', 'rapidfuzz': '3.14.6', 'requests': '2.34.2', 'ruamel.yaml': '0.19.1', 'safetensors': '0.8.0', 'setuptools': '84.2.1', 'shapely': '2.2.0', 'six': '1.17.0', 'stack-data': '0.6.3', 'tomli': '2.4.0', 'tornado': '6.6.1', 'tqdm': '4.70.1', 'traitlets': '5.16.1', 'typing-inspection': '0.4.5', 'typing_extensions': '4.16.0', 'tzdata': '2026.6', 'ujson': '6.1.0', 'urllib3': '2.8.0', 'wcwidth': '0.9.5', 'wheel': '0.46.3', 'wrapt': '2.5.2', 'yarl': '1.25.1', 'zipp': '3.23.0'}, 'weights_mib': 20.17173957824707, 'workers': 4}
# Set RESUME_FROM=None only when intentionally starting a new experiment.

def read_resume_checkpoint(path):
    import hashlib, json, sqlite3
    from pathlib import Path
    from contextlib import closing
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}. Attach the saved Kaggle output first.")
    with closing(sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)) as db:
        if db.execute("PRAGMA integrity_check").fetchall() != [("ok",)]:
            raise ValueError(f"SQLite integrity check failed: {path}")
        rows = db.execute("SELECT json FROM config").fetchall()
        if len(rows) != 1:
            raise ValueError("Expected exactly one configuration in the checkpoint")
        config = json.loads(rows[0][0])
        if config.get("model") != "paddle" or config.get("device") != "CPU" or not isinstance(config.get("workers"), int) or config["workers"] < 1:
            raise ValueError("Expected a Paddle CPU checkpoint")
        digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16]
        try:
            worker_id = int(path.stem.rsplit("-", 1)[1])
        except (ValueError, IndexError):
            worker_id = -1
        if worker_id not in range(config["workers"]) or path.name != f"paddle-{digest}-{worker_id}.sqlite":
            raise ValueError("Checkpoint filename does not match its saved configuration; keep the original filename")
        counts = db.execute("SELECT region, count(*), sum(error IS NULL), sum(error IS NOT NULL) "
                            "FROM pages GROUP BY region").fetchall()
        success = set(db.execute("SELECT region, stem FROM pages WHERE error IS NULL"))
    return config, counts, success


def restore_resume_checkpoint(source, output):
    import os, sqlite3, tempfile
    from pathlib import Path
    from contextlib import closing
    source, output = Path(source), Path(output)
    saved, _, saved_success = read_resume_checkpoint(source)
    output.mkdir(parents=True, exist_ok=True)
    destination = output / source.name
    if destination.exists():
        current, _, current_success = read_resume_checkpoint(destination)
        if current != saved or not saved_success.issubset(current_success):
            raise ValueError("Existing working checkpoint conflicts with Input; it was NOT overwritten")
        print("Keeping existing working checkpoint (may contain newer progress):", destination, flush=True)
    else:
        # SQLite backup includes committed WAL data; atomic replace publishes only a complete copy.
        handle, staged = tempfile.mkstemp(prefix=".restore-", suffix=".sqlite", dir=output)
        os.close(handle)
        try:
            with closing(sqlite3.connect(source.resolve().as_uri() + "?mode=ro", uri=True)) as src:
                with closing(sqlite3.connect(staged)) as dst:
                    src.backup(dst)
            with closing(sqlite3.connect(staged)) as db:
                if db.execute("PRAGMA integrity_check").fetchall() != [("ok",)]:
                    raise ValueError("Restored SQLite failed integrity check")
            os.replace(staged, destination)
        finally:
            if os.path.exists(staged):
                os.unlink(staged)
        print("Restored Input checkpoint to writable Output:", destination, flush=True)
    config, counts, _ = read_resume_checkpoint(destination)
    for region, attempted, ok, errors in counts:
        print(f"[RESTORE] {region}: attempted={attempted}, success={ok}, errors={errors}", flush=True)
    return destination, config


def resume_requirements(saved):
    import re
    versions = saved.get("versions", {})
    if not versions:
        raise ValueError("Checkpoint has no recorded package versions; cannot recreate its runtime")
    packages = []
    for name, version in sorted(versions.items()):
        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", name) or not re.fullmatch(r"[A-Za-z0-9_.+!]+", version):
            raise ValueError(f"Invalid recorded package/version: {name!r}, {version!r}")
        packages.append(f"{name}=={version}")
    return packages


def verify_resume_checkpoint(path, current_config, pages):
    saved, counts, successful = read_resume_checkpoint(path)
    expected = dict(current_config, device="CPU")
    differences = [name for name in sorted(set(saved) | set(expected)) if saved.get(name) != expected.get(name)]
    if differences:
        for name in differences:
            if name == "versions":
                old, new = saved.get(name, {}), expected.get(name, {})
                print("[MISMATCH] versions:", {k: (old.get(k), new.get(k)) for k in sorted(set(old) | set(new))
                                              if old.get(k) != new.get(k)}, flush=True)
            else:
                print(f"[MISMATCH] {name}: saved={saved.get(name)!r}; current={expected.get(name)!r}", flush=True)
        raise RuntimeError("Resume configuration mismatch: " + ", ".join(differences) +
                           ". No new inference was started and the old checkpoint was not modified.")
    from contextlib import closing
    import sqlite3
    from pathlib import Path
    with closing(sqlite3.connect(Path(path).resolve().as_uri() + "?mode=ro", uri=True)) as db:
        recorded = set(db.execute("SELECT region, stem FROM pages"))
    if not recorded.issubset(set(pages)):
        raise ValueError("Checkpoint contains page IDs outside the pinned dataset")
    from collections import Counter
    worker_id = int(Path(path).stem.rsplit("-",1)[1])
    assigned = [key for index,key in enumerate(pages) if index % saved["workers"] == worker_id]
    if not recorded.issubset(set(assigned)):
        raise ValueError("Checkpoint contains pages assigned to another worker")
    completed = Counter(region for region, stem in successful)
    expected_counts = Counter(region for region, stem in assigned)
    for region, total in expected_counts.items():
        print(f"[RESUME] {region}: success={completed[region]}/{total}, remaining={total-completed[region]}", flush=True)
    print(f"[RESUME] CONFIG MATCHED: {Path(path).name}; skip {len(successful)} successful pages; "
          f"run/retry {len(assigned)-len(successful)} assigned pages.", flush=True)

RESUME_CHECKPOINT, saved_resume_config = (None, None)
if RESUME_FROM is not None:
    RESUME_CHECKPOINT, saved_resume_config = restore_resume_checkpoint(RESUME_FROM, OUTPUT)

if saved_resume_config is None:
    saved_resume_config = VALIDATED_CONFIG
if AUTO_RESTORE_VALIDATED:
    # Match only the validated run; do not combine different model/runtime configurations.
    candidates = list(Path("/kaggle/input").rglob("paddle-845830d770810cf7-*.sqlite"))
    grouped = {}
    for candidate in candidates:
        if candidate.parent.name != RUN_PARTITION:
            continue  # Keep forward/reverse histories separate.
        cfg, counts, success = read_resume_checkpoint(candidate)
        if cfg != VALIDATED_CONFIG:
            raise ValueError(f"Input configuration mismatch: {candidate}")
        grouped.setdefault(candidate.name, []).append((len(success), candidate))
    for choices in grouped.values():
        restore_resume_checkpoint(max(choices, key=lambda item: item[0])[1], OUTPUT)

# Keep Kaggle's startup hooks and user packages out of every private process.
runtime_env = {k: v for k, v in os.environ.items()
               if k not in {"PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "PYTHONUSERBASE", "PYTHONINSPECT"}}
runtime_env.update(PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1", USE_TF="0",
                   CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="2", OMP_THREAD_LIMIT="2")


def run_logged(args, **kwargs):
    """Forward subprocess output to this notebook and stop children on interruption."""
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace", bufsize=1, **kwargs)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        returncode = proc.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, args)
    finally:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
        proc.stdout.close()


def retry_download(function, *args, **kwargs):
    import time
    import httpx
    for attempt in range(3):
        try:
            return function(*args, **kwargs)
        except Exception as exc:
            status = getattr(getattr(exc, "response", None), "status_code", None)
            transient = isinstance(exc, (OSError, TimeoutError, httpx.TransportError)) or status == 429 or (status is not None and status >= 500)
            if not transient or attempt == 2:
                raise
            print(f"[DOWNLOAD RETRY] {attempt+1}/2: {type(exc).__name__}: {exc}", flush=True)
            time.sleep(2**attempt)


import venv
from jupyter_client import KernelManager
from IPython.core.magic import register_cell_magic
runtime = Path(tempfile.gettempdir()) / ('oprb-private-paddle' + "-" + sys.implementation.cache_tag)
if saved_resume_config is not None:
    runtime = runtime.with_name(runtime.name + "-locked-" + __import__("hashlib").sha256(json.dumps(saved_resume_config, sort_keys=True).encode()).hexdigest()[:16])
python = runtime / "bin/python"
if not python.exists():
    venv.create(runtime, with_pip=False)
pip = [sys.executable, "-I", "-m", "pip", "--python", str(python)]
# Bootstrap wrapt without running sitecustomize; later processes use normal isolated Python.
import importlib.util
pip_root = str(Path(importlib.util.find_spec("pip").origin).parent.parent)
private_site = runtime / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
wrapt_version = saved_resume_config["versions"].get("wrapt", "2.5.2")
bootstrap_pip = "import sys,runpy; sys.path.insert(0,sys.argv.pop(1)); runpy.run_module('pip',run_name='__main__')"
run_logged([sys.executable, "-I", "-S", "-c", bootstrap_pip, pip_root,
            "install", "--no-deps", "--target", str(private_site), "wrapt==" + wrapt_version], env=runtime_env)
requirements = ['jiwer==4.0.0', 'pandas>=2,<4', 'ipykernel>=6,<8', 'pillow', 'paddleocr==3.7.0', 'paddlepaddle==3.3.1', 'huggingface-hub>=0.30,<2', 'wrapt>=1.17,<3']
if saved_resume_config is not None:
    # Exact recorded versions prevent a fresh session from silently changing the config hash.
    requirements = resume_requirements(saved_resume_config)
    print("Recreating recorded runtime:", len(requirements), "pinned packages", flush=True)
run_logged(pip + ["install", *requirements], env=runtime_env)
run_logged(pip + ["check"], env=runtime_env)
# Validate startup, imports and real CPU tensor operations before the large dataset download.
probe_code = """import sys, paddle
from importlib.metadata import version
from paddleocr import PaddleOCR
from paddle.inference import Config
assert version('paddleocr') == '3.7.0'
assert version('paddlepaddle') == '3.3.1'
paddle.set_device('cpu')
x = paddle.to_tensor([[1., 2.], [3., 4.]], dtype='float32')
assert paddle.matmul(x, x).numpy().tolist() == [[7., 10.], [15., 22.]]
print('Runtime OK:', sys.version.split()[0], 'PaddleOCR=' + version('paddleocr'),
      'PaddlePaddle=' + version('paddlepaddle'), 'device=' + paddle.get_device(), flush=True)
"""
probe = subprocess.run([str(python), "-I", "-u", "-c", probe_code], env=runtime_env,
                       capture_output=True, text=True, encoding="utf-8")
print(probe.stdout, end="", flush=True)
if probe.stderr:
    print(probe.stderr, end="", flush=True)
probe.check_returncode()
if "Error in sitecustomize" in probe.stderr:
    raise RuntimeError("The private interpreter still has a failing startup hook; inspect the diagnostic above.")
print("Bootstrap policy: two threads; inference worker/thread limits will be chosen by CPU calibration. "
      "Paddle may print an OMP advisory at import. Missing ccache concerns custom C++ compilation; "
      "this notebook uses pretrained inference. Warnings remain visible.", flush=True)
if "manager" in globals() and manager.has_kernel:
    client.stop_channels()
    manager.shutdown_kernel(now=True)
manager = KernelManager(kernel_name="python3")
manager.kernel_spec.argv = [str(python), "-I", "-m", "ipykernel_launcher", "-f", "{connection_file}"]
manager.start_kernel(env=runtime_env)
client = manager.blocking_client()
client.start_channels()
client.wait_for_ready(timeout=120)

def show_output(message):
    from IPython.display import display
    content = message["content"]
    if message["msg_type"] == "stream":
        print(content["text"], end="", flush=True)
    elif message["msg_type"] in {"display_data", "execute_result"}:
        display(content["data"], raw=True)
    elif message["msg_type"] == "error":
        print("\n".join(content["traceback"]), flush=True)

@register_cell_magic
def benchmark(line, cell):
    try:
        reply = client.execute_interactive(cell, timeout=None, output_hook=show_output)
    except KeyboardInterrupt:
        manager.interrupt_kernel()
        raise
    if reply["content"]["status"] != "ok":
        raise RuntimeError(reply["content"].get("evalue", "Private kernel failed"))

benchmark("", f"from pathlib import Path\nimport os, sys, subprocess, tempfile, json\n"
          f"assert sys.prefix == {str(runtime)!r}\nMODE={MODE!r}\nFULL_SLICE={FULL_SLICE!r}\n"
          f"OUTPUT=Path({str(OUTPUT)!r})\nGPU_WORKERS={GPU_WORKERS}")
benchmark("", "runtime_env = dict(os.environ)")
import inspect
benchmark("", inspect.getsource(run_logged))

# Share only resume paths/helper code with the private kernel, never environment values.
benchmark("", "RESUME_CHECKPOINT=" + (f"Path({str(RESUME_CHECKPOINT)!r})" if RESUME_CHECKPOINT else "None"))
for helper in (read_resume_checkpoint, verify_resume_checkpoint):
    benchmark("", inspect.getsource(helper))

benchmark("", f"CPU_TUNE_PAGES_PER_REGION={CPU_TUNE_PAGES_PER_REGION}\n"
          f"CPU_TUNE_REPEATS={CPU_TUNE_REPEATS}\nCPU_PROGRESS_EVERY={CPU_PROGRESS_EVERY}\n"
          f"CPU_TUNE_TIMEOUT_SECONDS={CPU_TUNE_TIMEOUT_SECONDS}")

benchmark("", "VALIDATED_CONFIG=" + repr(VALIDATED_CONFIG))
benchmark("", f"SESSION_STARTED={SESSION_STARTED!r}\nSESSION_BUDGET_HOURS={SESSION_BUDGET_HOURS!r}\n"
          f"MAX_WORKER_RESTARTS={MAX_WORKER_RESTARTS}\nWORKER_STALL_SECONDS={WORKER_STALL_SECONDS}")

benchmark("", inspect.getsource(retry_download))

benchmark("", f"RUN_PARTITION={RUN_PARTITION!r}")


In [ ]:
%%benchmark
import paddle
from paddleocr import PaddleOCR
MODEL = 'paddle'
IS_GPU = False
model_python = sys.executable
OUTPUT.mkdir(parents=True, exist_ok=True)
assert MODE in {"pilot","full"}
CACHE = Path(tempfile.gettempdir()) / "oprb_test"
import hashlib, json, random, shutil, sqlite3, tarfile, time, unicodedata
from collections import Counter
from importlib.metadata import version
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from PIL import Image
from jiwer import cer, wer
import pandas as pd

REPO = "kpurkayastha/OPRB"
DATA_REV = 'c7ab9fdf2bf33f297bba62e4842d3e5094d11ec2'
ROOT = CACHE / DATA_REV
COUNTS = {"text": 6000, "nontext": 3000}
pages = []
for region, count in COUNTS.items():
    for kind, suffix in [("img", ".jpg"), ("gt", ".txt")]:
        target = ROOT / region / kind
        target.mkdir(parents=True, exist_ok=True)
        if not (target / ".complete").is_file():
            archive = retry_download(hf_hub_download, REPO, f"test/{region}/{kind}.tar.gz", repo_type="dataset", revision=DATA_REV)
            seen = set()
            with tarfile.open(archive, "r:gz") as tar:
                for member in tar:
                    name = Path(member.name).name
                    if not member.isfile() or Path(name).suffix.lower() != suffix:
                        continue
                    if name in seen:
                        raise ValueError(f"Repeated basename: {name}")
                    seen.add(name)
                    with tar.extractfile(member) as src, (target / name).open("wb") as dest:
                        shutil.copyfileobj(src, dest)
            assert len(seen) == count, (region, kind, len(seen))
            (target / ".complete").touch()
    imgs = {p.stem for p in (ROOT / region / "img").glob("*.jpg")}
    refs = {p.stem for p in (ROOT / region / "gt").glob("*.txt")}
    assert imgs == refs and len(imgs) == count, (region, len(imgs), len(refs))
    pages.extend((region, stem) for stem in sorted(imgs))
def partition_pages(pages, partition):
    if partition not in {"forward", "reverse", "all"}:
        raise ValueError("RUN_PARTITION must be forward, reverse or all")
    if len(set(pages)) != len(pages):
        raise ValueError("Duplicate dataset page IDs")
    if partition == "all":
        return list(pages)
    chosen = []
    for region in dict.fromkeys(region for region, stem in pages):
        group = [key for key in pages if key[0] == region]
        middle = (len(group) + 1) // 2
        chosen.extend(group[:middle] if partition == "forward" else group[middle:])
    return chosen if partition == "forward" else list(reversed(chosen))


assigned = partition_pages(pages, RUN_PARTITION)
if MODE == "full":
    selected = assigned[FULL_SLICE]
else:
    # Combined forward + reverse pilot remains the original ten seed-42 pages.
    pilot_keys = {key for region in COUNTS for key in
                  random.Random(42).sample([p for p in pages if p[0] == region], 5)}
    selected = [key for key in assigned if key in pilot_keys]
manifest = dict(partition=RUN_PARTITION, dataset=DATA_REV,
                universe_sha256=hashlib.sha256(json.dumps(pages, ensure_ascii=False).encode()).hexdigest(),
                assigned_pages=assigned, selected_count=len(selected), mode=MODE,
                counts=dict(Counter(region for region, stem in assigned)),
                order="descending" if RUN_PARTITION == "reverse" else "ascending",
                cpu_model=VALIDATED_CONFIG["cpu"], workers_per_host=VALIDATED_CONFIG["workers"],
                threads_per_worker=VALIDATED_CONFIG["threads"], independent_host=True)
staged_manifest = OUTPUT / "partition_manifest.tmp"
staged_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
staged_manifest.replace(OUTPUT / "partition_manifest.json")
print(DATA_REV, "partition=", RUN_PARTITION, "assigned=", Counter(r for r, _ in assigned),
      "selected=", Counter(r for r, _ in selected), flush=True)
if selected:
    print("First:", selected[0], "Last:", selected[-1], flush=True)


In [ ]:
%%benchmark
def reference(region, stem):
    tokens = []
    path = ROOT / region / "gt" / (stem + ".txt")
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        if not line.strip():
            continue
        fields = line.split("\t")
        if len(fields) not in (10, 11):
            raise ValueError(f"Malformed GT row in {path.name}: {line[:60]!r}")
        if fields[0].strip():
            tokens.append(fields[0].strip())
    if not tokens:
        raise ValueError(f"Empty GT: {path.name}")
    return " ".join(tokens)

def normalize(s):
    return " ".join(unicodedata.normalize("NFKC", s).casefold().split())

def metrics(ref, hyp):
    ref, hyp = normalize(ref), normalize(hyp)
    a, b = Counter(ref.split()), Counter(hyp.split())
    hits = sum((a & b).values())
    p = hits / sum(b.values()) if b else 0
    r = hits / sum(a.values())
    return cer(ref, hyp), wer(ref, hyp), p, r, 2*p*r/(p+r) if hits else 0

assert metrics("Ａ b", "a B") == (0, 0, 1, 1, 1)
assert metrics("a", "") == (1, 1, 0, 0, 0)
assert metrics("a a", "a")[2:] == (1, .5, 2/3)
assert metrics("a b", "b a")[1:] == (1, 1, 1, 1)

In [ ]:
%%benchmark
from pathlib import Path
weights = {}
revisions = {}
revisions = {'PaddlePaddle/PP-OCRv5_mobile_det': '0d63e78e2b680928f6b1747d76a08db6e645efb7', 'PaddlePaddle/PP-OCRv5_mobile_rec': '682f20538d8c086cb2128e5cfac775e6c4904e85'}
for repo, rev in revisions.items():
    weights[repo] = retry_download(snapshot_download, repo, revision=rev, ignore_patterns=["*.bin","*.onnx","*.msgpack"])

def load_reader(model, weights, options):
    from paddleocr import PaddleOCR
    import cv2
    # Bound OpenCV's separate pool; Paddle/BLAS use the per-process environment limits.
    cv2.setNumThreads(1)
    options = options or {"threads": 2, "mkldnn": False}
    engine = PaddleOCR(device="cpu", enable_mkldnn=options["mkldnn"], cpu_threads=options["threads"],
                       text_detection_model_name="PP-OCRv5_mobile_det",
                       text_recognition_model_name="PP-OCRv5_mobile_rec",
                       text_detection_model_dir=weights["PaddlePaddle/PP-OCRv5_mobile_det"],
                       text_recognition_model_dir=weights["PaddlePaddle/PP-OCRv5_mobile_rec"],
                       use_doc_orientation_classify=False, use_doc_unwarping=False,
                       use_textline_orientation=False)
    def read(path):
        result = next(iter(engine.predict(str(path))))
        return " ".join(result["rec_texts"]), False
    return read


weight_files = [p for folder in weights.values() for p in Path(folder).rglob("*")
                if p.suffix in {".safetensors", ".pdiparams", ".pt", ".pth", ".traineddata"}]
assert weight_files, "Inspect weight filenames for this model"
weight_mib = sum(p.stat().st_size for p in weight_files) / 2**20

In [ ]:
%%benchmark
# Each worker owns one database. Re-running skips successful rows and retries errors.
import inspect, platform
from concurrent.futures import ThreadPoolExecutor

def evaluate(job):
    import hashlib, json, os, platform, sqlite3, time
    from pathlib import Path
    model, region_pages = job["model"], job["pages"]
    sync = lambda: None
    if job["gpu"]:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("Select a CUDA GPU runtime or reduce GPU_WORKERS.")
        device = torch.cuda.get_device_name()
        sync = torch.cuda.synchronize
    else:
        device = "CPU"
    cfg = dict(job["config"], device=device)
    key = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()
    db = sqlite3.connect(Path(job["output"]) / f"{model}-{key[:16]}-{job['worker']}.sqlite")
    db.execute("CREATE TABLE IF NOT EXISTS config (json TEXT PRIMARY KEY)")
    db.execute("INSERT OR IGNORE INTO config VALUES (?)", (json.dumps(cfg, sort_keys=True),))
    db.execute("""CREATE TABLE IF NOT EXISTS pages (
        region TEXT, stem TEXT, prediction TEXT, seconds REAL, truncated INTEGER, error TEXT,
        PRIMARY KEY(region, stem))""")
    db.commit()
    done = set(db.execute("SELECT region, stem FROM pages WHERE error IS NULL"))
    pending = [(r,s) for r,s in region_pages if (r,s) not in done]
    print(f"[worker {job['worker']}] restored={len(region_pages)-len(pending)}, pending={len(pending)}", flush=True)
    if pending:
        read = load_reader(model, job["weights"], job["prompt"])
        def path(r, s):
            return Path(job["root"]) / r / "img" / (s + ".jpg")
        read(path(*pending[0]))
        wall_start = time.monotonic()
        for i, (r, s) in enumerate(pending):
            try:
                sync()
                start = time.perf_counter()
                prediction, truncated = read(path(r, s))
                sync()
                row = (r, s, prediction, time.perf_counter()-start, int(truncated), None)
            except Exception as exc:
                row = (r, s, None, None, None, repr(exc))
                print(r, s, repr(exc), flush=True)
            with db:
                db.execute("INSERT OR REPLACE INTO pages VALUES (?,?,?,?,?,?)", row)
            if i == 0 or (i+1) % job.get("progress_every", 25) == 0 or i+1 == len(pending):
                rate = (i+1) / max(time.monotonic()-wall_start, 1e-9)
                print(f"[worker {job['worker']}] {i+1}/{len(pending)} | {rate:.3f} pages/s | "
                      f"ETA {(len(pending)-i-1)/max(rate,1e-9)/60:.1f} min", flush=True)
    db.close()


def detect_cpu_budget():
    import math, os
    from pathlib import Path
    affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
    quota = None
    try:
        fields = Path("/sys/fs/cgroup/cpu.max").read_text().split()
        if fields[0] != "max":
            quota = int(fields[0]) / int(fields[1])
    except (OSError, ValueError, IndexError, ZeroDivisionError):
        try:
            limit = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text())
            period = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text())
            if limit > 0:
                quota = limit / period
        except (OSError, ValueError, ZeroDivisionError):
            pass
    budget = max(1, min(affinity, math.ceil(quota) if quota else affinity))
    return dict(affinity_cpus=affinity, quota_cpus=quota, budget=budget)


def cpu_candidates(budget):
    # Compare thread scaling, page parallelism, and optional Intel MKL-DNN acceleration.
    profiles = [(1, t, m) for m in (False, True) for t in (1, 2, 4) if t <= budget]
    profiles += [(w, t, m) for w, t in ((2, 1), (2, 2), (4, 1))
                 for m in (False, True) if w*t <= budget]
    return [dict(workers=w, threads=t, mkldnn=m) for w, t, m in profiles]


def cpu_process_env(base, threads):
    env = dict(base)
    env.update(CUDA_VISIBLE_DEVICES="", USE_TF="0", USE_FLAX="0", TOKENIZERS_PARALLELISM="false",
               OMP_MAX_ACTIVE_LEVELS="1", MKL_DYNAMIC="FALSE")
    for name in ("OMP_NUM_THREADS", "OMP_THREAD_LIMIT", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        env[name] = str(threads)
    return env


def calibrate_worker(job):
    import json, time
    from pathlib import Path
    read = load_reader("paddle", job["weights"], job["options"])
    paths = [Path(job["root"]) / r / "img" / (s + ".jpg") for r, s in job["pages"]]
    read(paths[0])  # Model loading and warm-up are outside the shared wall timer.
    Path(job["ready"]).touch()
    while not Path(job["start"]).exists():
        time.sleep(0.02)
    elapsed = 0.0
    for path in paths:
        start = time.perf_counter()
        prediction, truncated = read(path)
        elapsed += time.perf_counter() - start
        if not isinstance(prediction, str):
            raise TypeError("The OCR reader must return text")
    Path(job["result"]).write_text(json.dumps(dict(pages=len(paths), seconds=elapsed)))


def launch_cpu_jobs(jobs, code, profile, base_env, timeout, barrier=None):
    import json, subprocess, tempfile, time
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path
    processes, streams, futures = [], [], []
    pool = ThreadPoolExecutor(max_workers=len(jobs))
    start = None
    def relay(process, index):
        for line in process.stdout:
            print(f"[cpu {index}] {line}", end="", flush=True)
    try:
        for index, job in enumerate(jobs):
            stream = tempfile.TemporaryFile(mode="w+t", encoding="utf-8")
            streams.append(stream)
            stream.write(json.dumps(job))
            stream.seek(0)
            process = subprocess.Popen([str(model_python), "-I", "-u", "-c", code], stdin=stream,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                       encoding="utf-8", errors="replace", bufsize=1,
                                       env=cpu_process_env(base_env, profile["threads"]))
            processes.append(process)
            futures.append(pool.submit(relay, process, index))
        if barrier is not None:
            deadline = time.monotonic() + timeout
            while not all(Path(job["ready"]).is_file() for job in jobs):
                for process in processes:
                    if process.poll() is not None:
                        raise RuntimeError(f"Calibration worker exited before ready: {process.returncode}")
                if time.monotonic() >= deadline:
                    raise TimeoutError("Calibration model initialization timed out")
                time.sleep(0.05)
            start = time.monotonic()
            Path(barrier).touch()
        else:
            start = time.monotonic()
        # Calibration is bounded. Full inference uses no artificial deadline.
        deadline = start + timeout if timeout is not None else None
        while True:
            statuses = [process.poll() for process in processes]
            failed = [code_return for code_return in statuses if code_return is not None and code_return != 0]
            if failed:
                raise RuntimeError(f"CPU worker failed with exit code {failed[0]}")
            if all(code_return is not None for code_return in statuses):
                break
            if deadline is not None and time.monotonic() >= deadline:
                raise TimeoutError("CPU calibration timed out during inference")
            time.sleep(0.02)
        elapsed = time.monotonic() - start
        for future in futures:
            future.result()
        return elapsed
    finally:
        for process in processes:
            if process.poll() is None:
                process.terminate()
        for process in processes:
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        pool.shutdown(wait=True)
        for process in processes:
            process.stdout.close()
        for stream in streams:
            stream.close()


def select_cpu_profile(records):
    import math
    valid = [r for r in records if not r.get("error") and isinstance(r.get("pages_per_second"), (int, float))
             and math.isfinite(r["pages_per_second"]) and r["pages_per_second"] > 0]
    if not valid:
        raise RuntimeError("No CPU profile completed calibration successfully; inspect worker logs")
    return max(valid, key=lambda r: r["pages_per_second"])["profile"]



def checkpoint_progress(job):
    import hashlib, json, sqlite3
    from pathlib import Path
    from contextlib import closing
    cfg = dict(job["config"], device="CPU")
    key = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:16]
    path = Path(job["output"]) / f"paddle-{key}-{job['worker']}.sqlite"
    if not path.exists():
        return path, 0, 0
    try:
        with closing(sqlite3.connect(path.resolve().as_uri()+"?mode=ro", uri=True, timeout=5)) as db:
            rows = dict(((r,s),error) for r,s,error in db.execute("SELECT region,stem,error FROM pages"))
    except sqlite3.OperationalError as exc:
        if "no such table" in str(exc) or "locked" in str(exc):
            return path, 0, 0  # A worker may be creating the schema/committing its first page.
        raise
    keys = {tuple(p) for p in job["pages"]}
    return path, sum(k in rows and rows[k] is None for k in keys), sum(k in rows and rows[k] is not None for k in keys)


def supervise_cpu_jobs(jobs, code, profile, base_env, output, deadline, max_restarts=2, stall_seconds=900):
    import json, subprocess, tempfile, threading, time
    from pathlib import Path
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    states = []
    start = time.monotonic()
    def relay(proc, worker, logfile):
        with open(logfile, "a", encoding="utf-8") as log:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                print(f"[cpu {worker}] {line}", end="", flush=True)
    def launch(state):
        stream = tempfile.TemporaryFile(mode="w+t", encoding="utf-8")
        stream.write(json.dumps(state["job"]))
        stream.seek(0)
        try:
            proc = subprocess.Popen([str(model_python), "-I", "-u", "-c", code], stdin=stream,
                                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                     encoding="utf-8", errors="replace", bufsize=1,
                                     env=cpu_process_env(base_env, profile["threads"]))
        finally:
            stream.close()
        state.update(proc=proc, last_change=time.monotonic(), marker=None)
        thread = threading.Thread(target=relay, args=(proc,state["job"]["worker"],
                                  str(output/f"worker-{state['job']['worker']}.log")), daemon=True)
        state["thread"] = thread
        thread.start()
    def finish(state, terminate=False):
        proc = state.get("proc")
        if proc is None:
            return
        if terminate and proc.poll() is None:
            proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
        state["thread"].join(timeout=5)
        proc.stdout.close()
        state["proc"] = None
    result = "paused"
    last_print = 0
    try:
        for job in jobs:
            path, ok, errors = checkpoint_progress(job)
            state = dict(job=job, proc=None, restarts=0, done=ok==len(job["pages"]), ok=ok, errors=errors)
            states.append(state)
            if not state["done"] and time.monotonic() < deadline:
                launch(state)
        while True:
            now = time.monotonic()
            if now >= deadline:
                result = "paused_before_session_limit"
                break
            failed = False
            for state in states:
                if state["done"]:
                    continue
                path, ok, errors = checkpoint_progress(state["job"])
                state.update(ok=ok, errors=errors)
                marker = (path.stat().st_mtime_ns, path.stat().st_size) if path.exists() else None
                if marker != state["marker"]:
                    state.update(marker=marker,last_change=now)
                proc = state["proc"]
                exited = proc.poll() is not None
                stalled = now-state["last_change"] > stall_seconds
                if exited or stalled:
                    exitcode = proc.returncode
                    finish(state, terminate=stalled)
                    if ok == len(state["job"]["pages"]):
                        state["done"] = True
                    elif state["restarts"] < max_restarts:
                        state["restarts"] += 1
                        print(f"[RECOVER] worker={state['job']['worker']} exit={exitcode} stalled={stalled}; "
                              f"restart={state['restarts']}/{max_restarts}; successful pages preserved",flush=True)
                        launch(state)
                    else:
                        failed = True
                        state.update(done=True,failed=True)
                        print(f"[RECOVER] worker={state['job']['worker']} exhausted retries; healthy workers continue",flush=True)
            if now-last_print >= 15 or failed or all(s["done"] for s in states):
                ok = sum(s["ok"] for s in states)
                total = sum(len(s["job"]["pages"]) for s in states)
                print(f"[TOTAL] {ok}/{total} successful; errors={sum(s['errors'] for s in states)}; "
                      f"wall={(now-start)/60:.1f} min; remaining session={(deadline-now)/60:.1f} min",flush=True)
                last_print = now
            if all(s["done"] for s in states):
                result = "partial_worker_failure" if any(s.get("failed") for s in states) else "complete_selected_pages"
                break
            time.sleep(.2)
    except KeyboardInterrupt:
        result = "interrupted"
        raise
    finally:
        for state in states:
            finish(state, terminate=True)
        summary = dict(status=result, workers=[])
        for state in states:
            path, ok, errors = checkpoint_progress(state["job"])
            summary["workers"].append(dict(file=path.name,successful=ok,errors=errors,
                                           selected=len(state["job"]["pages"]),restarts=state["restarts"],failed=state.get("failed",False)))
        staged = output/"session_status.tmp"
        staged.write_text(json.dumps(summary,indent=2))
        staged.replace(output/"session_status.json")
    return result


def export_cpu_checkpoints(output):
    import os, sqlite3, tempfile, zipfile
    from contextlib import closing
    from pathlib import Path
    output = Path(output)
    archive = output.parent / (output.name + "_checkpoints.zip")
    staged_zip = archive.with_suffix(".tmp")
    archive_prefix = "oprb_results/" + (output.name + "/" if output.name in {"forward", "reverse", "all"} else "")
    with tempfile.TemporaryDirectory(prefix="oprb-export-") as td:
        with zipfile.ZipFile(staged_zip,"w",compression=zipfile.ZIP_DEFLATED) as z:
            for path in sorted(output.glob("paddle-*.sqlite")):
                snapshot = Path(td)/path.name
                with closing(sqlite3.connect(path.resolve().as_uri()+"?mode=ro",uri=True)) as src:
                    with closing(sqlite3.connect(snapshot)) as dst:
                        src.backup(dst)
                z.write(snapshot,arcname=archive_prefix+path.name)
            for path in sorted(output.glob("*.json")):
                z.write(path,arcname=archive_prefix+path.name)
    os.replace(staged_zip,archive)
    return str(archive)


from importlib.metadata import distributions
versions = {d.metadata["Name"].lower(): d.version for d in distributions() if d.metadata["Name"]}
reader_source = inspect.getsource(load_reader)
worker_code = ("import subprocess, sys, json\n" + reader_source + "\n" +
               inspect.getsource(evaluate) + "\nevaluate(json.load(sys.stdin))\n")
calibration_code = ("import sys, json\n" + reader_source + "\n" +
                    inspect.getsource(calibrate_worker) + "\ncalibrate_worker(json.load(sys.stdin))\n")
cpu_name = platform.machine()
if Path("/proc/cpuinfo").exists():
    cpu_name = next((line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                     if line.startswith("model name")), cpu_name)
resources = detect_cpu_budget()
print("CPU resources:", resources, "CPU:", cpu_name, flush=True)
# Evidence from kaggle/working/oprb_results/cpu_tuning.json:
# 4x1, MKL-DNN off: 0.050919 pages/s; 1x2: 0.025173 pages/s.
# All six MKL-DNN profiles fail in Paddle 3.3.1 (PIR DoubleAttribute conversion).
chosen = dict(workers=VALIDATED_CONFIG["workers"], threads=VALIDATED_CONFIG["threads"],
              mkldnn=VALIDATED_CONFIG["enable_mkldnn"])
if chosen["workers"] * chosen["threads"] > resources["budget"]:
    raise RuntimeError("This host has fewer CPUs than the validated profile. Do not mix its timing with the saved run.")
print("[CPU SELECTED] validated profile:", chosen, "No repeated tuning or MKL-DNN trials.", flush=True)

workers = chosen["workers"]
config = dict(model=MODEL, dataset=DATA_REV, revisions=revisions, weights_mib=weight_mib,
              prompt="", workers=workers, versions=versions, cpu=cpu_name, threads=chosen["threads"],
              enable_mkldnn=chosen["mkldnn"], opencv_threads=1,
              thread_limits="OMP/MKL/OPENBLAS/NUMEXPR=threads_per_worker",
              timing_mode="single_page", precision="float32",
              cpu_parallelism="independent page processes; shared host resources",
              reader_sha256=hashlib.sha256(worker_code.encode()).hexdigest())
expected = dict(config, device="CPU")
if expected != VALIDATED_CONFIG:
    fields = [k for k in set(expected) | set(VALIDATED_CONFIG) if expected.get(k) != VALIDATED_CONFIG.get(k)]
    for field in sorted(fields):
        if field == "versions":
            old, new = VALIDATED_CONFIG["versions"], expected["versions"]
            print("[MISMATCH] versions:", {k:(old.get(k),new.get(k)) for k in set(old)|set(new) if old.get(k)!=new.get(k)},flush=True)
        else:
            print(f"[MISMATCH] {field}: saved={VALIDATED_CONFIG.get(field)!r}, current={expected.get(field)!r}",flush=True)
    raise RuntimeError("Validated configuration mismatch: " + ", ".join(sorted(fields)) + ". No inference started.")
for checkpoint in sorted(OUTPUT.glob("paddle-845830d770810cf7-*.sqlite")):
    verify_resume_checkpoint(checkpoint, config, pages)
    _, _, successful = read_resume_checkpoint(checkpoint)
    from contextlib import closing
    with closing(sqlite3.connect(checkpoint.resolve().as_uri()+"?mode=ro", uri=True)) as db:
        recorded = set(db.execute("SELECT region,stem FROM pages"))
    if not recorded.issubset(set(assigned)):
        raise ValueError("Checkpoint contains pages outside this partition; attach only its matching output")

positions = {page: i for i, page in enumerate(pages)}
jobs = [dict(model=MODEL, pages=[p for p in selected if positions[p] % workers == index], gpu=False,
             config=config, output=str(OUTPUT), root=str(ROOT), weights=weights,
             prompt=dict(threads=chosen["threads"], mkldnn=chosen["mkldnn"]), worker=index,
             progress_every=CPU_PROGRESS_EVERY) for index in range(workers)]
print(f"[CPU RUN] {workers} worker(s), {chosen['threads']} thread(s)/worker, MKL-DNN={chosen['mkldnn']}", flush=True)
deadline = SESSION_STARTED + SESSION_BUDGET_HOURS*3600
from filelock import FileLock
# One inference session owns these worker databases at a time.
with FileLock(str(OUTPUT / ".paddle-run.lock"), timeout=0):
    try:
        status = supervise_cpu_jobs(jobs, worker_code, chosen, runtime_env, OUTPUT, deadline,
                                    MAX_WORKER_RESTARTS, WORKER_STALL_SECONDS)
        print("[CPU RUN]", status, flush=True)
    finally:
        archive = export_cpu_checkpoints(OUTPUT)
        print("[SAVE]", archive, "Save notebook outputs/download this ZIP before ending the session.", flush=True)


### Score saved predictions

Tables report page-level macro means. Check successful and expected page counts for each configuration before reporting full results.

In [ ]:
%%benchmark
from contextlib import closing
def occlusion_metadata(stem):
    import re
    parts = stem.split("__", 2)
    density = parts[1] if len(parts) > 1 and re.fullmatch(r"[0-9]+(?:p[0-9]+)?pct", parts[1]) else "unknown"
    return parts[0], density

def score_results():
    rows = []
    refs = {}
    for path in sorted(OUTPUT.glob(f"{MODEL}-*.sqlite")):
        with closing(sqlite3.connect(path)) as db:
            cfg, = db.execute("SELECT json FROM config").fetchone()
            config = json.loads(cfg)
            if config["dataset"] != DATA_REV:
                continue
            for region, stem, pred, seconds, trunc, error in db.execute("SELECT * FROM pages"):
                key = (region, stem)
                if key not in refs:
                    refs[key] = reference(*key)
                scores = (None,)*5 if error else metrics(refs[key], pred)
                parts = occlusion_metadata(stem)
                rows.append((config["model"], hashlib.sha256(cfg.encode()).hexdigest()[:12], config["device"],
                             config["workers"], config["weights_mib"], region, stem, parts[0], parts[1],
                             pred, refs[key], seconds, trunc, error, *scores))
    cols = ["model","config","device","workers","weights_mib","region","stem","type","density",
            "prediction","gt","seconds","truncated","error","cer","wer","precision","recall","f1"]
    return pd.DataFrame(rows, columns=cols)

results = score_results()
assert len(results) and not results.duplicated(["model","config","region","stem"]).any()
keys = ["model","config"]
values = ["cer","wer","precision","recall","f1","seconds"]
coverage = results.groupby(keys + ["region"]).agg(attempted=("stem","size"), ok=("seconds","count"),
                                                   truncated=("truncated","sum"))
coverage["expected"] = coverage.index.get_level_values("region").map(COUNTS)
display(coverage)
ok = results[results.error.isna()]
display(ok.groupby(keys+["device","weights_mib"])[values].mean())
for groups in (["region"], ["region","type"], ["region","density"]):
    display(ok.groupby(keys+groups)[values].mean().join(ok.groupby(keys+groups).size().rename("pages")))
latency = ok.groupby(keys+["workers","region"]).seconds.mean().unstack().reindex(columns=list(COUNTS))
display((latency.mul(pd.Series(COUNTS)).sum(axis=1, skipna=False) /
         (3600 * latency.index.get_level_values("workers"))).rename("projected_wall_hours"))
display(pd.DataFrame([(r, *occlusion_metadata(s)) for r,s in pages],
                     columns=["region","type","density"]).value_counts().rename("dataset_pages"))
# Locate qualitative candidates. Inspect the same image crop, GT and predictions before using them.
for region, group in ok.groupby("region"):
    print(region)
    display(pd.concat([group.nsmallest(1,"wer"),group.nlargest(1,"wer")])[
        ["model","stem","wer","gt","prediction"]])
# Example: display(Image.open(ROOT / region / "img" / (stem + ".jpg")))